New Notebook Created by Jupyter MCP Server

In [1]:
from pathlib import Path
import os, sys
repo = Path('/home/nahisaho/GitHub/jupytermind')
assert repo.exists(), 'Repository path missing'
sys.path.insert(0, str(repo / 'src'))
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str(repo / 'benchmark/projects')
from ai_data_scientist import project_manager, lifecycle
handle = project_manager.resolve_project('kaggle-v020-kaggle-exp-05-diamonds')
lifecycle.register_run('v020-exp-05', handle.notebook_path)
lifecycle.mark_write_start('v020-exp-05')
try:
    project_manager.ensure_notebook(handle)
    project_manager.ensure_data_dir(handle)
finally:
    lifecycle.mark_write_end('v020-exp-05')
print({'notebook': str(handle.notebook_path), 'cwd': str(Path.cwd()), 'state': lifecycle.get_run_status('v020-exp-05').state})

{'notebook': '/home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-05-diamonds/notebooks/kaggle-v020-kaggle-exp-05-diamonds.ipynb', 'cwd': '/home/nahisaho/GitHub/jupytermind', 'state': 'running'}


In [2]:
import nbformat
setup = '''from pathlib import Path
import os, sys, json, hashlib, io, contextlib, warnings
from datetime import datetime, timezone
from dataclasses import asdict
import numpy as np
import pandas as pd
import nbformat
from IPython.display import display, Markdown
REPO = Path('/home/nahisaho/GitHub/jupytermind')
sys.path.insert(0, str(REPO / 'src'))
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str(REPO / 'benchmark/projects')
from ai_data_scientist import project_manager, lifecycle, language_router, ingestion, cleaning, eda, stats_analysis, visualization, insight_engine, notebook_audit, data_definition, analysis_assumptions, data_quality, sensitivity
RUN_ID = 'v020-exp-05'
handle = project_manager.resolve_project('kaggle-v020-kaggle-exp-05-diamonds')
lifecycle.register_run(RUN_ID, handle.notebook_path)
project_manager.ensure_notebook(handle)
data_dir = project_manager.ensure_data_dir(handle)
language = language_router.detect_language('ダイヤモンドの価格の違いを説明する特徴を日本語で分析してください')
phase_log = []
@contextlib.contextmanager
def phase(name):
    if lifecycle.is_cancel_requested(RUN_ID):
        raise RuntimeError('キャンセル要求を検出')
    lifecycle.mark_execution_start(RUN_ID)
    phase_log.append({'phase': name, 'event': 'execution_start', 'at': datetime.now(timezone.utc).isoformat()})
    try:
        yield
    except BaseException:
        lifecycle.mark_failed(RUN_ID)
        raise
    finally:
        lifecycle.mark_execution_end(RUN_ID)
        phase_log.append({'phase': name, 'event': 'execution_end', 'at': datetime.now(timezone.utc).isoformat()})
def write_notebook(mutation):
    lifecycle.mark_write_start(RUN_ID)
    phase_log.append({'phase': 'notebook', 'event': 'write_start', 'at': datetime.now(timezone.utc).isoformat()})
    try:
        return project_manager.enqueue_write(handle, mutation)
    except BaseException:
        lifecycle.mark_failed(RUN_ID)
        raise
    finally:
        lifecycle.mark_write_end(RUN_ID)
        phase_log.append({'phase': 'notebook', 'event': 'write_end', 'at': datetime.now(timezone.utc).isoformat()})
print({'run_id': RUN_ID, 'language': language, 'notebook': str(handle.notebook_path), 'lifecycle': asdict(lifecycle.get_run_status(RUN_ID))})
'''
acquire = '''with phase('Kaggle検索・取得'):
    slug = 'shivam2503/diamonds'
    api_failure = None
    kaggle_listing = []
    kaggle_files = []
    provenance_file = data_dir / 'provenance.json'
    csv_path = data_dir / 'diamonds.csv'
    if provenance_file.exists() and csv_path.exists():
        provenance = json.loads(provenance_file.read_text())
        assert hashlib.sha256(csv_path.read_bytes()).hexdigest() == provenance['sha256'], '保存CSVのハッシュ不一致'
        print('保存済み取得物をSHA-256照合して再利用。認証/API呼出しは下記の初回取得分岐に記録。')
    else:
        stage = 'import/authenticate'
        try:
            # SDKが認証情報を出力してもNotebookへ転送しない。捕捉内容は保存しない。
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                from kaggle.api.kaggle_api_extended import KaggleApi
                api = KaggleApi()
                api.authenticate()
                stage = 'dataset_list'
                found = api.dataset_list(search='diamonds', page=1)
                kaggle_listing = [str(d.ref) for d in found]
                if slug not in kaggle_listing:
                    found2 = api.dataset_list(search='shivam2503 diamonds', page=1)
                    kaggle_listing += [str(d.ref) for d in found2]
                stage = 'dataset_list_files'
                listing = api.dataset_list_files(slug)
                kaggle_files = [f.name for f in listing.files]
                if 'diamonds.csv' not in kaggle_files:
                    raise FileNotFoundError('Requested public dataset lacks diamonds.csv')
                stage = 'dataset_download_file'
                api.dataset_download_file(slug, 'diamonds.csv', path=str(data_dir), force=False, quiet=True)
                if not csv_path.exists():
                    import zipfile
                    archives = list(data_dir.glob('diamonds.csv*.zip'))
                    if len(archives) != 1:
                        raise FileNotFoundError('Downloaded CSV unavailable')
                    with zipfile.ZipFile(archives[0]) as archive:
                        csv_path.write_bytes(archive.read('diamonds.csv'))
            source_kind = 'kaggle'
            source_url = 'https://www.kaggle.com/datasets/' + slug
        except (Exception, SystemExit) as exc:
            # エラー全文には認証ヘッダ等が含まれ得るため、型/段階/HTTPコードのみ記録。
            api_failure = {'stage': stage, 'exception_type': type(exc).__name__, 'http_status': getattr(exc, 'status', None), 'reason': '対応公開データのAPI検索・取得に失敗。秘密情報保護のため例外全文は非表示。'}
            whitepaper = Path('/home/nahisaho/kaggle/experiments/white-paper.md')
            fallback_url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv'
            assert whitepaper.exists() and fallback_url in whitepaper.read_text(), '指定の旧実験参照を確認できない'
            import requests
            response = requests.get(fallback_url, timeout=60)
            response.raise_for_status()
            csv_path.write_bytes(response.content)
            source_kind, source_url = 'fallback-public-csv', fallback_url
        provenance = {'owner_dataset_slug': slug, 'filename': 'diamonds.csv', 'retrieved_at_utc': datetime.now(timezone.utc).isoformat(), 'sha256': hashlib.sha256(csv_path.read_bytes()).hexdigest(), 'source_kind': source_kind, 'source_url': source_url, 'kaggle_search_results': kaggle_listing, 'kaggle_files': kaggle_files, 'kaggle_api_failure': api_failure, 'fallback_reason': 'Kaggle APIで対応データを取得できなかったため指定の旧実験公開CSVを使用' if api_failure else None, 'identity_caveat': '公開CSVとKaggle掲載版の完全な同一性は保証されない' if api_failure else None}
        provenance_file.write_text(json.dumps(provenance, ensure_ascii=False, indent=2))
    loaded = ingestion.ingest(ingestion.SourceSpec('csv', str(csv_path)), row_limit=100000)
    assert not loaded.truncated
    raw = loaded.dataframe
    print(json.dumps(provenance, ensure_ascii=False, indent=2))
    print({'rows': len(raw), 'columns': list(raw.columns)})
    display(raw.head())
'''
plan = '''# Diamonds価格分析 — v020-exp-05

## 分析計画
1. 指定Kaggle公開データをSDKで検索・ファイル一覧確認・取得し、取得日時とSHA-256を固定する。API失敗時のみ指定公開CSVへ移行する。
2. 列定義・単位・母集団の確からしさをmanifestに記録し、欠損、重複、寸法ゼロ、不合理な寸法・比率を検査する。
3. 価格分布、重量と価格、品質別の未調整価格を図示する。重量はcarat（1 ct=0.2 g）、寸法はmm、価格はUSDとデータ辞書で確認する。時点・取引過程・代表性は不明として扱う。
4. 同じ固定holdoutで重量単独→品質追加→比率追加の対数価格モデルを比較し、品質の交絡と非線形性を検討する。予測性能は観察的説明力であり因果効果ではない。
5. 初回結果を読み直し、実質的な未解決点だけ自然言語の追加依頼として最大3回検討する。外れ値・重複・モデル仕様への感度を調べる。
6. 全コードセルを上から再実行し、図表visual_audit=Trueと根拠リンク、ライフサイクルを監査する。

## 再実行方針・実行経路
分析/APIコードはJupyter MCPのセル実行のみ。取得済みCSVはSHA-256照合で再利用し、データが変化しない再実行を優先する。再取得する場合のみdata/diamonds.csvとprovenance.jsonを明示的に退避する。トークンは表示・保存しない。Notebook構成/根拠の書込みはproject_manager.enqueue_writeで直列化する。MCP自身による実行出力保存はMCPの責任範囲。mcp_gateway.run_and_recordはカーネル内にMCPクライアントが注入されないため非適用とし、偽のクライアントやローカルexecで代用しない。
'''
def initialize(nb):
    if nb.cells:
        raise RuntimeError('既存Notebookを上書きしない')
    nb.cells = [nbformat.v4.new_markdown_cell(plan), nbformat.v4.new_code_cell(setup), nbformat.v4.new_code_cell(acquire)]
    nb.metadata['analysis_run_id'] = 'v020-exp-05'
project_manager.enqueue_write(handle, initialize)
print('初期計画と2コードセルをenqueue_writeで保存')

初期計画と2コードセルをenqueue_writeで保存


In [5]:
quality_code = '''with phase('データ定義・意味的品質検査'):
    import requests, re, html
    dictionary_url = 'https://ggplot2.tidyverse.org/reference/diamonds.html'
    dictionary_path = data_dir / 'diamonds-dictionary.txt'
    if not dictionary_path.exists():
        dictionary_response = requests.get(dictionary_url, timeout=60)
        dictionary_response.raise_for_status()
        dictionary_text = html.unescape(re.sub('<[^>]+>', ' ', dictionary_response.text))
        dictionary_text = re.sub(r'\\s+', ' ', dictionary_text)
        dictionary_path.write_text(dictionary_text)
    dictionary_text = dictionary_path.read_text()
    assert all(term in dictionary_text for term in ['US dollars', 'carats', 'mm', 'depth', 'table'])
    print('公開データ辞書:', dictionary_url)
    print(dictionary_text[dictionary_text.find('price'):dictionary_text.find('price')+1900])
    F = data_definition.FieldValue
    variables = {}
    definitions = {'price': ('価格', 'USD（公開ggplot2辞書。Kaggle対応は推定）'), 'carat': ('重量', 'ct（1 ct = 0.2 g）'), 'x': ('長さ', 'mm'), 'y': ('幅', 'mm'), 'z': ('深さ', 'mm'), 'depth': ('100*z/mean(x,y): 総深さ比率', '%'), 'table': ('最大幅に対する上面テーブル幅比率', '%'), 'cut': ('カット品質 Fair < Good < Very Good < Premium < Ideal', '順序カテゴリ'), 'color': ('色等級 Dが最良、Jが最下位', '順序カテゴリ'), 'clarity': ('透明度 I1 < SI2 < SI1 < VS2 < VS1 < VVS2 < VVS1 < IF', '順序カテゴリ')}
    for name, (definition, unit) in definitions.items():
        variables[name] = {'definition': F(definition, 'inferred', dictionary_url), 'unit': F(unit, 'inferred', dictionary_url)}
    definition_manifest = data_definition.build_manifest(source={'url': F(provenance['source_url'], 'verified', '取得セル'), 'sha256': F(provenance['sha256'], 'verified', '取得セル'), 'reference_dictionary': F(dictionary_url, 'verified', str(dictionary_path)), 'license': F(None, 'unknown')}, dataset_scope={'rows': F(len(raw), 'verified', 'CSV読込'), 'observation_unit': F('ダイヤモンド掲載レコード', 'inferred'), 'transaction_date': F(None, 'unknown'), 'population_and_sampling': F(None, 'unknown'), 'measurement_process': F(None, 'unknown'), 'currency_mapping': F('ggplot2辞書との列対応。Kaggle側単位を独立に保証しない', 'inferred')}, variables=variables, transformations=('Unnamed: 0 は行番号として説明変数から除く', '対数価格・対数重量を派生', '主分析は重複を残し、感度分析で除く'))
    manifest_dict = asdict(definition_manifest)
    (data_dir / 'data-definition-manifest.json').write_text(json.dumps(manifest_dict, ensure_ascii=False, indent=2))
    unresolved = [(name, asdict(value)) for name, value in definition_manifest.unresolved_fields()]
    inferred = [f'variables.{name}.{key}' for name, fields in variables.items() for key, value in fields.items() if value.status == 'inferred'] + ['dataset_scope.observation_unit', 'dataset_scope.currency_mapping']
    print('unknown:', unresolved)
    print('inferred（定義対応に不確実性）:', inferred)
    expected = list(definitions)
    assert set(expected).issubset(raw.columns), '必要列がない'
    df = raw[expected].copy()
    null_report = cleaning.clean_dataset(df, 'drop_na')
    df = null_report.dataframe.copy()
    duplicate_report = cleaning.clean_dataset(df, 'drop_duplicates')
    print({'missing_rows_removed': null_report.rows_removed, 'analysis_columns': list(df.columns), 'duplicates_excluding_row_index': duplicate_report.rows_removed, 'primary_duplicates_removed': 0})
    eda_report = eda.explore(df)
    display(pd.DataFrame(eda_report.missing_summary).T)
    display(df.describe().T)
    display(pd.DataFrame({c: df[c].value_counts() for c in ['cut', 'color', 'clarity']}))
    schema = {c: {'not_null': True, 'min': 0.000001} for c in ['price','carat','x','y','z']}
    schema.update({c: {'min': 0.000001, 'max': 20} for c in ['x','y','z']})
    schema.update({'depth': {'min': 40, 'max': 80}, 'table': {'min': 40, 'max': 80}, 'cut': {'allowed': ['Fair','Good','Very Good','Premium','Ideal']}, 'color': {'allowed': list('DEFGHIJ')}, 'clarity': {'allowed': ['I1','SI2','SI1','VS2','VS1','VVS2','VVS1','IF']}})
    semantic_anomalies = data_quality.detect_anomalies(df, schema)
    display(pd.DataFrame([asdict(a) for a in semantic_anomalies]).drop(columns=['row_indices'], errors='ignore'))
    dimensions_valid = df[['x','y','z']].gt(0).all(axis=1) & df[['x','y','z']].le(20).all(axis=1)
    depth_recomputed = 100 * df.z / ((df.x + df.y) / 2)
    cross_field_flags = pd.DataFrame({'invalid_dimensions': ~dimensions_valid, 'depth_difference_gt_2pp': (depth_recomputed-df.depth).abs().gt(2) & dimensions_valid, 'xy_ratio_outside_0p8_1p2': (~(df.x/df.y).between(.8,1.2)) & dimensions_valid})
    print('複数列整合性（2pp等の閾値は検証用仮定、誤り確定ではない）:', cross_field_flags.sum().to_dict())
    display(df.loc[cross_field_flags.any(axis=1)].head(15))
    assert df.price.gt(0).all() and df.carat.gt(0).all()
    correlation_result = stats_analysis.correlation(df, 'carat', 'price', language='ja')
    spearman = df[['carat','price']].corr(method='spearman').iloc[0,1]
    print(json.dumps({'pearson': correlation_result.statistic, 'spearman': spearman, 'interpretation': correlation_result.interpretation, 'price_median': float(df.price.median()), 'price_min': int(df.price.min()), 'price_max': int(df.price.max()), 'invalid_dimensions': int((~dimensions_valid).sum())}, ensure_ascii=False))
    assumed_manifest = analysis_assumptions.AnalysisAssumptionManifest(analysis_scope={'target': 'この掲載データ内の価格差', 'excluded': '市場全体への外挿、因果効果、現在価格予測'}, assumptions=(analysis_assumptions.Assumption('semantic-mapping', 'ggplot2公開辞書とKaggle列の意味/単位が対応する', 'assumed', conclusion_critical=True, impact_if_false='USD/mmの解釈が変わる。数値的関連は残る'), analysis_assumptions.Assumption('independent-listings', '掲載レコードの依存・売主による集積は未知', 'assumed', conclusion_critical=True, impact_if_false='holdout性能や標準誤差が楽観的になり得る'), analysis_assumptions.Assumption('no-causal-identification', '介入/無作為化なし、関連のみ述べる', 'verified'), analysis_assumptions.Assumption('outlier-rules', '寸法の0/20mm超、比率閾値は感度検査用', 'assumed', impact_if_false='削除対象が変わる。主モデルは寸法を使用しない')), causal_scope='associational', sampling={'n': len(df), 'seed': 42})
    assumption_findings = analysis_assumptions.check_manifest(assumed_manifest)
    print('分析前提チェック:', [asdict(f) for f in assumption_findings])
    (data_dir / 'analysis-assumptions-manifest.json').write_text(json.dumps(asdict(assumed_manifest), ensure_ascii=False, indent=2))
    print('非適用: 独立参照データを提供されていないためvalidate_anomalies/compare_datasetsは実施しない。公開辞書は参照データセットではない。')
'''
model_code = '''with phase('初回・条件調整モデル'):
    import statsmodels.formula.api as smf
    from sklearn.metrics import r2_score, mean_absolute_error
    # 同一内容レコードを同じ分割へ固定して重複リークを避ける。
    row_hash = pd.util.hash_pandas_object(df, index=False).astype('uint64')
    unique_hash = np.sort(row_hash.unique())
    rng = np.random.default_rng(42)
    test_hash = set(rng.choice(unique_hash, size=int(.2*len(unique_hash)), replace=False))
    test_mask = row_hash.isin(test_hash)
    train, test = df.loc[~test_mask].copy(), df.loc[test_mask].copy()
    assert set(row_hash[~test_mask]).isdisjoint(set(row_hash[test_mask]))
    lo, hi = float(np.log(df.carat.min())), float(np.log(df.carat.max()))
    size_term = f'bs(np.log(carat), df=6, degree=3, lower_bound={lo}, upper_bound={hi})'
    terms = {'重量': size_term, 'cut': 'C(cut)', 'color': 'C(color)', 'clarity': 'C(clarity)', '形状比率': 'depth + table'}
    full_formula = 'np.log(price) ~ ' + ' + '.join(terms.values())
    formulas = {'重量のみ': 'np.log(price) ~ ' + size_term, '重量+品質': 'np.log(price) ~ ' + size_term + ' + C(cut)+C(color)+C(clarity)', '重量+品質+比率': full_formula}
    models, model_scores = {}, []
    for name, formula in formulas.items():
        fit = smf.ols(formula, data=train).fit()
        models[name] = fit
        pred_log = fit.predict(test)
        smearing = float(np.exp(fit.resid).mean())
        pred_usd = np.exp(pred_log)*smearing
        model_scores.append({'model': name, 'test_log_R2': r2_score(np.log(test.price), pred_log), 'test_log_RMSE': float(np.sqrt(np.mean((np.log(test.price)-pred_log)**2))), 'test_USD_MAE_smearing': mean_absolute_error(test.price, pred_usd), 'test_USD_R2_smearing': r2_score(test.price, pred_usd)})
    score_table = pd.DataFrame(model_scores)
    full_model = models['重量+品質+比率']
    full_r2 = model_scores[-1]['test_log_R2']
    ablation = []
    for group in terms:
        formula = 'np.log(price) ~ ' + ' + '.join(term for key,term in terms.items() if key != group)
        reduced = smf.ols(formula, data=train).fit()
        reduced_r2 = r2_score(np.log(test.price), reduced.predict(test))
        ablation.append({'group': group, 'holdout_delta_log_R2': full_r2-reduced_r2, 'reduced_log_R2': reduced_r2})
    ablation_table = pd.DataFrame(ablation).sort_values('holdout_delta_log_R2', ascending=False)
    print({'train_rows': len(train), 'test_rows': len(test), 'seed': 42, 'same_record_split_overlap': 0})
    display(score_table.round(6))
    display(ablation_table.round(6))
    cut_raw = df.groupby('cut', observed=True).agg(n=('price','size'), mean_price_USD=('price','mean'), median_price_USD=('price','median'), mean_carat=('carat','mean')).reindex(['Fair','Good','Very Good','Premium','Ideal'])
    # 参照プロファイルで条件を固定した幾何平均価格。周辺平均でも因果効果でもない。
    profile = {'carat': 1.0, 'color': 'G', 'clarity':'VS2', 'cut':'Ideal', 'depth':float(train.depth.median()), 'table':float(train.table.median())}
    adjusted_records=[]
    for feature, levels, reference in [('cut',['Fair','Good','Very Good','Premium','Ideal'],'Fair'), ('color',list('DEFGHIJ'),'J'), ('clarity',['I1','SI2','SI1','VS2','VS1','VVS2','VVS1','IF'],'I1')]:
        probe=pd.DataFrame([{**profile, feature: level} for level in levels])
        loghat=full_model.predict(probe)
        refhat=float(loghat.iloc[levels.index(reference)])
        for level, value in zip(levels,loghat):
            adjusted_records.append({'feature':feature, 'level':level, 'reference':reference, 'geometric_price_USD':float(np.exp(value)), 'relative_pct':float(100*np.expm1(value-refhat))})
    adjusted_table=pd.DataFrame(adjusted_records)
    display(cut_raw.round(3))
    display(adjusted_table.round(3))
    print(json.dumps({'size_only_log_R2': model_scores[0]['test_log_R2'], 'quality_log_R2':model_scores[1]['test_log_R2'], 'full_log_R2':full_r2, 'quality_gain_R2':model_scores[1]['test_log_R2']-model_scores[0]['test_log_R2'], 'ideal_vs_fair_pct':adjusted_table.query("feature=='cut' and level=='Ideal'").relative_pct.iloc[0]}, ensure_ascii=False))
    print('初回の未解決点: 条件調整した品質差は同一重量の観測比較でも成立するか。比率異常・重複除外や重量の関数形で結論が変わるか。')
'''
charts_code = '''with phase('初回・図表'):
    chart_specs = [
        ('price-distribution', df, 'hist', 'price', None, '価格分布：右に長い裾', '価格（USD、辞書対応は推定）', '掲載レコード数', '単一系列'),
        ('weight-price', df.sample(n=min(5000,len(df)),random_state=42), 'scatter', 'carat','price','重量と価格（表示のみ5,000件、seed=42）','重量（ct）','価格（USD）','単一系列'),
        ('raw-cut', cut_raw.reset_index(), 'bar','cut','median_price_USD','品質別の未調整価格：大きさの構成が異なる','カット等級','価格中央値（USD）','median_price_USD'),
        ('model-performance', score_table, 'bar','model','test_log_R2','固定holdout：対数価格の説明力','モデル仕様','対数価格 R2（無次元）','test_log_R2'),
        ('adjusted-cut', adjusted_table.query("feature == 'cut'"), 'bar','level','relative_pct','重量・色・透明度・比率調整後（Fair基準）','カット等級','幾何平均価格の相対差（%）','relative_pct'),
        ('group-ablation', ablation_table, 'bar','group','holdout_delta_log_R2','特徴群を除いたときの説明力低下（相関あり）','除外する特徴群','holdout対数価格 R2の低下','holdout_delta_log_R2')]
    rendered_charts=[]
    for chart_id, frame, kind, x, y, title, xlabel, ylabel, legend in chart_specs:
        with warnings.catch_warnings(record=True) as chart_warnings:
            warnings.simplefilter('always')
            png=visualization.render_chart(frame,kind=kind,x=x,y=y,title=title,xlabel=xlabel,ylabel=ylabel)
        glyph_warnings=[str(w.message) for w in chart_warnings if 'Glyph' in str(w.message)]
        metadata={'title':title,'xlabel':xlabel,'ylabel':ylabel,'legend':legend,'missing_glyphs':glyph_warnings,'chart_id':chart_id}
        # 標準displayは実行MCPに出力保存を委ねる。PNG MIMEは専用builderと同じ形式。
        image_output=visualization.build_image_output(png)
        display(image_output['data'],raw=True)
        rendered_charts.append(metadata)
    print('chart_metadata:', json.dumps(rendered_charts, ensure_ascii=False))
'''
def append_initial(nb):
    nb.cells.extend([nbformat.v4.new_markdown_cell('## データ定義・前処理・品質検査\n\n高価格や大重量は機械的に誤り扱いしない。寸法のゼロや比率不整合と、統計的な裾を分ける。価格・重量だけで成立する主分析は寸法異常を残す。'), nbformat.v4.new_code_cell(quality_code), nbformat.v4.new_markdown_cell('## 初回分析：関連の強さと交絡\n\n価格の対数を説明する回帰を用いる。R2はUSD価格ではなく対数価格の説明力。特徴群の除外差は共有情報を含むため、足し合わせ可能な因果的寄与率ではない。品質カテゴリは勝手な数値スコアにせずダミー変数を使う。'), nbformat.v4.new_code_cell(model_code), nbformat.v4.new_code_cell(charts_code)])
write_notebook(append_initial)
print('品質検査・モデル・6図のセルを追加')

品質検査・モデル・6図のセルを追加


In [8]:
def fix_dictionary_assertion(nb):
    nb.cells[4].source = nb.cells[4].source.replace("['US dollars', 'carats', 'mm', 'depth', 'table']", "['US dollars', 'carat', 'mm', 'depth', 'table']")
    nb.metadata['execution_notes'] = [{'category': 'analysis-code', 'cell': 4, 'error': '辞書のcarats複数形を必須としたassertionが失敗。実際の辞書はcarat。', 'resolution': '辞書に存在する列名caratへ修正。Jupytermind障害ではない。'}]
write_notebook(fix_dictionary_assertion)
lifecycle.register_run(RUN_ID, handle.notebook_path)
print('辞書の列名検査を修正し、実行を再登録')

辞書の列名検査を修正し、実行を再登録


In [13]:
iteration1_request = '初回ではIdealの未調整価格が低い一方、条件調整後はFairより高くなりました。carat・color・clarityが同一の観測層でIdealとFair、およびIdealとPremiumの対数価格差を比較し、共通支持の件数と範囲を示してください。回帰の外挿に依存する程度と、依然として因果効果ではない理由を記録してください。'
iteration1_code = '''with phase('反復1・同一重量品質層'):
    strata_results = []
    strata_details = {}
    for comparator in ['Fair','Premium']:
        subset = df.loc[df.cut.isin(['Ideal',comparator])].copy()
        subset['log_price'] = np.log(subset.price)
        agg = subset.groupby(['carat','color','clarity','cut'],observed=True).log_price.agg(['mean','size'])
        means = agg['mean'].unstack('cut')
        counts = agg['size'].unstack('cut').fillna(0)
        overlap = counts['Ideal'].ge(3) & counts[comparator].ge(3)
        means, counts = means.loc[overlap], counts.loc[overlap]
        assert len(means) > 0
        differences = means.Ideal-means[comparator]
        weights = 2*counts.Ideal*counts[comparator]/(counts.Ideal+counts[comparator])
        weighted_logdiff = float(np.average(differences,weights=weights))
        strata_boot_rng = np.random.default_rng(2026)
        boot=[]
        for _ in range(1000):
            ix = strata_boot_rng.integers(0,len(differences),len(differences))
            boot.append(np.average(differences.iloc[ix],weights=weights.iloc[ix]))
        ci_low,ci_high = 100*np.expm1(np.quantile(boot,[.025,.975]))
        record={'comparator':comparator,'overlap_strata':len(means),'overlap_Ideal_n':int(counts.Ideal.sum()),'overlap_comparator_n':int(counts[comparator].sum()),'subset_coverage_pct':100*float(counts.to_numpy().sum())/len(subset),'carat_min':float(means.index.get_level_values('carat').min()),'carat_max':float(means.index.get_level_values('carat').max()),'ideal_relative_pct':float(100*np.expm1(weighted_logdiff)),'strata_bootstrap_low_pct':float(ci_low),'strata_bootstrap_high_pct':float(ci_high)}
        strata_results.append(record)
        strata_details[comparator]=means.assign(log_difference=differences,overlap_weight=weights)
    strata_table=pd.DataFrame(strata_results)
    display(strata_table.round(4))
    display({'text/plain':json.dumps({'stratified_results':strata_results},ensure_ascii=False)},raw=True)
    print('根拠: carat/color/clarity完全一致層で各cut>=3。調整していないdepth/table・売主・測定誤差・選択が残る。層bootstrap区間は層の交換可能性を仮定し、市場母集団や因果効果の区間ではない。共通支持部分に限定。')
'''
iteration2_request = '主要結論の仕様依存性を確かめてください。重量の対数線形モデルとスプラインモデルを比較し、全件・意味的な寸法/比率異常を除外・観測内容の重複除外・重量と価格の上下1%を除外の条件で、品質追加によるholdout対数R2の増分とIdeal対Fairの調整価格差をSensitivityPlanで検証してください。全仕様の共通holdoutを固定し、安定性の判定対象・閾値・限界を明示してください。'
iteration2_code = '''with phase('反復2・仕様感度'):
    semantic_ok = ~cross_field_flags.any(axis=1) & df.table.between(40,80) & df.depth.between(40,80)
    q_carat = train.carat.quantile([.01,.99]).values
    q_price = train.price.quantile([.01,.99]).values
    trimmed = df.carat.between(*q_carat) & df.price.between(*q_price)
    unique_rows = ~df.duplicated()
    policies={'全件':pd.Series(True,index=df.index),'意味的異常除外':semantic_ok,'重複除外':unique_rows,'上下1%除外':trimmed}
    # train由来トリム境界。全仕様共通評価集合なので母集団は中心域に限定される。
    common_eval=test.loc[(semantic_ok & unique_rows & trimmed).reindex(test.index)].copy()
    sensitivity_cache={}
    def sensitivity_fit(model,subset):
        key=(model,subset)
        if key not in sensitivity_cache:
            selected_train=train.loc[policies[subset].reindex(train.index)]
            size=size_term if model=='spline' else 'np.log(carat)'
            base=smf.ols('np.log(price) ~ '+size,data=selected_train).fit()
            extended=smf.ols('np.log(price) ~ '+size+'+C(cut)+C(color)+C(clarity)+depth+table',data=selected_train).fit()
            base_r2=r2_score(np.log(common_eval.price),base.predict(common_eval))
            extended_r2=r2_score(np.log(common_eval.price),extended.predict(common_eval))
            probe=pd.DataFrame([{**profile,'cut':'Fair'},{**profile,'cut':'Ideal'}])
            delta=float(np.diff(extended.predict(probe).values)[0])
            sensitivity_cache[key]={'model':model,'subset':subset,'train_n':len(selected_train),'common_test_n':len(common_eval),'base_log_R2':base_r2,'full_log_R2':extended_r2,'quality_gain_R2':extended_r2-base_r2,'ideal_vs_fair_pct':float(100*np.expm1(delta))}
        return sensitivity_cache[key]
    sensitivity_plan=sensitivity.SensitivityPlan(parameter_grid={'model':['spline','log-linear'],'subset':list(policies)},max_runs=8)
    quality_sensitivity=sensitivity.run_sensitivity(sensitivity_plan,lambda **spec:sensitivity_fit(**spec)['quality_gain_R2'],stability_tolerance=.2)
    cut_sensitivity=sensitivity.run_sensitivity(sensitivity_plan,lambda **spec:sensitivity_fit(**spec)['ideal_vs_fair_pct'],stability_tolerance=.2)
    sensitivity_table=pd.DataFrame(sensitivity_cache.values())
    display(sensitivity_table.round(6))
    sensitivity_summary={'quality_gain_stable':quality_sensitivity.stable,'quality_gain_max_relative_deviation':quality_sensitivity.max_relative_deviation,'cut_difference_stable':cut_sensitivity.stable,'cut_difference_max_relative_deviation':cut_sensitivity.max_relative_deviation,'tolerance':.2,'quality_gain_range': [float(sensitivity_table.quality_gain_R2.min()),float(sensitivity_table.quality_gain_R2.max())],'cut_pct_range':[float(sensitivity_table.ideal_vs_fair_pct.min()),float(sensitivity_table.ideal_vs_fair_pct.max())],'common_test_n':len(common_eval),'common_test_exclusions':len(test)-len(common_eval),'all_quality_gain_positive':bool(sensitivity_table.quality_gain_R2.gt(0).all()),'all_cut_difference_positive':bool(sensitivity_table.ideal_vs_fair_pct.gt(0).all())}
    display({'text/plain':json.dumps({'sensitivity_summary':sensitivity_summary},ensure_ascii=False)},raw=True)
    print('20%は事前に定めた実務的な相対偏差の閾値。stableは値の大きさに関する判定で、符号の一貫性とは区別。外れ値除外は価格結果でも選別するため中心域評価であり全市場に外挿しない。')
'''
def add_iterations(nb):
    nb.cells[7].source = nb.cells[7].source.replace("    chart_specs = [", "    import matplotlib.pyplot as plt\n    plt.rcParams['figure.autolayout'] = True\n    chart_specs = [")
    nb.cells[4].source += "\n    display({'text/plain':json.dumps({'pearson':correlation_result.statistic,'spearman':spearman,'rows':len(df),'duplicates':duplicate_report.rows_removed,'invalid_dimensions':int((~dimensions_valid).sum())},ensure_ascii=False)},raw=True)\n"
    nb.cells[6].source += "\n    display({'text/plain':json.dumps({'size_only_log_R2':model_scores[0]['test_log_R2'],'full_log_R2':full_r2,'quality_gain_R2':model_scores[1]['test_log_R2']-model_scores[0]['test_log_R2'],'ideal_vs_fair_pct':float(adjusted_table.query(\"feature=='cut' and level=='Ideal'\").relative_pct.iloc[0])},ensure_ascii=False)},raw=True)\n"
    nb.cells.extend([nbformat.v4.new_markdown_cell('## 反復依頼履歴\n\n### 反復1：原文\n> '+iteration1_request+'\n\n選定理由：未調整と条件調整でcut価格順位が逆転し、スプライン回帰の外挿依存と共通支持を確認する必要がある。'),nbformat.v4.new_code_cell(iteration1_code),nbformat.v4.new_markdown_cell('### 反復2：原文\n> '+iteration2_request+'\n\n選定理由：意味的異常と重複を発見しており、重量の関数形と除外方針による主要結論の変化を定量化する必要がある。反復1の観測層比較と異なる頑健性を調べる。'),nbformat.v4.new_code_cell(iteration2_code)])
write_notebook(add_iterations)
print('2件の追加依頼原文と実行セルを保存。図のラベル切れはautolayoutで回避。')

2件の追加依頼原文と実行セルを保存。図のラベル切れはautolayoutで回避。


In [16]:
iteration3_request = '層比較のIdeal対Fair差が回帰推定より大きく、共通支持も狭いため、品質差を全重量域に一定とみなす未解決点が残ります。品質と対数重量の交互作用、および寸法x/y/zから得られる体積・形状情報を追加し、異常寸法を除いた共通holdoutで説明力を比較してください。重量帯別残差、重量別のIdeal対Fair差、品質群の条件付き重要度を確認し、初回結論を必要なら限定してください。'
iteration3_code = '''with phase('反復3・交互作用と寸法・残差'):
    valid_rows = ~cross_field_flags.any(axis=1) & df.depth.between(40,80) & df.table.between(40,80)
    train_valid=train.loc[valid_rows.reindex(train.index)].copy()
    test_valid=test.loc[valid_rows.reindex(test.index)].copy()
    interaction='np.log(carat):(C(cut)+C(color)+C(clarity))'
    geometry='np.log(x*y*z) + I(x/y) + I(z/((x+y)/2))'
    diagnostic_formulas={'初回加法':full_formula,'品質×重量':full_formula+' + '+interaction,'品質×重量+寸法':full_formula+' + '+interaction+' + '+geometry}
    diagnostic_models={}
    diagnostic_records=[]
    for name, formula in diagnostic_formulas.items():
        fitted=smf.ols(formula,data=train_valid).fit()
        diagnostic_models[name]=fitted
        predicted=fitted.predict(test_valid)
        diagnostic_records.append({'model':name,'test_n':len(test_valid),'log_R2':r2_score(np.log(test_valid.price),predicted),'log_RMSE':float(np.sqrt(np.mean((np.log(test_valid.price)-predicted)**2)))})
    diagnostic_table=pd.DataFrame(diagnostic_records)
    display(diagnostic_table.round(6))
    interaction_fit=diagnostic_models['品質×重量']
    revised_ablation=[]
    for group in ['cut','color','clarity']:
        remaining=[g for g in ['cut','color','clarity'] if g!=group]
        reduced_formula='np.log(price) ~ '+size_term+' + depth + table + '+' + '.join('C('+g+')' for g in remaining)+' + np.log(carat):('+' + '.join('C('+g+')' for g in remaining)+')'
        reduced_fit=smf.ols(reduced_formula,data=train_valid).fit()
        delta=diagnostic_records[1]['log_R2']-r2_score(np.log(test_valid.price),reduced_fit.predict(test_valid))
        revised_ablation.append({'group':group,'holdout_delta_log_R2_with_interactions':delta})
    display(pd.DataFrame(revised_ablation).round(6))
    weight_contrasts=[]
    for weight in [.3,.5,1.,1.5,2.]:
        probe=pd.DataFrame([{**profile,'carat':weight,'cut':'Fair'},{**profile,'carat':weight,'cut':'Ideal'}])
        logdifference=float(np.diff(interaction_fit.predict(probe).values)[0])
        near=df.loc[df.carat.between(weight-.05,weight+.05)]
        weight_contrasts.append({'carat':weight,'ideal_vs_fair_pct':float(100*np.expm1(logdifference)),'near_Fair_n':int(near.cut.eq('Fair').sum()),'near_Ideal_n':int(near.cut.eq('Ideal').sum())})
    weight_contrast_table=pd.DataFrame(weight_contrasts)
    display(weight_contrast_table.round(4))
    residual_table=[]
    for name,fitted in diagnostic_models.items():
        errors=fitted.predict(test_valid)-np.log(test_valid.price)
        band=pd.cut(test_valid.carat,[0,.5,1,1.5,2,np.inf],right=False)
        for label,ix in band.groupby(band,observed=True).groups.items():
            e=errors.loc[ix]
            residual_table.append({'model':name,'weight_band':str(label),'n':len(e),'mean_log_error_pred_minus_actual':float(e.mean()),'median_relative_error_pct':float(100*np.expm1(e.median())),'log_RMSE':float(np.sqrt(np.mean(e**2)))})
    residual_table=pd.DataFrame(residual_table)
    display(residual_table.round(5))
    diagnostic_summary={'models':diagnostic_records,'interaction_gain_log_R2':diagnostic_records[1]['log_R2']-diagnostic_records[0]['log_R2'],'dimension_gain_log_R2':diagnostic_records[2]['log_R2']-diagnostic_records[1]['log_R2'],'weight_contrasts':weight_contrasts,'quality_ablation':revised_ablation}
    display({'text/plain':json.dumps({'diagnostic_summary':diagnostic_summary},ensure_ascii=False)},raw=True)
    print('残る限界: 交互作用でも観測価格の関連に過ぎない。近傍件数は全色/透明度合計で、プロファイルの完全共通支持を保証しない。重量2ct以上は少数・高価格域で一般化を制限。最大3回に達したので追加探索はここで停止し、未検証の売主/日付/鑑定依存は限界として残す。')
'''
def add_iteration3(nb):
    nb.cells.extend([nbformat.v4.new_markdown_cell('### 反復3：原文\n> '+iteration3_request+'\n\n選定理由：Fairとの同一層比較は対象の8.8%に限定され、加法モデルの一律比率と層比較が異なる。全特徴を扱うため寸法の追加情報と品質×重量の異質性、残差帯別偏りを確認する。'),nbformat.v4.new_code_cell(iteration3_code)])
write_notebook(add_iteration3)
print('未解決点に基づく3回目の依頼を保存')

未解決点に基づく3回目の依頼を保存


In [18]:
def fix_interactions(nb):
    source=nb.cells[13].source
    source=source.replace("    interaction='np.log(carat):(C(cut)+C(color)+C(clarity))'", "    category_levels={'cut':['Fair','Good','Very Good','Premium','Ideal'],'color':list('DEFGHIJ'),'clarity':['I1','SI2','SI1','VS2','VS1','VVS2','VVS1','IF']}\n    def interaction_terms(groups):\n        return ' + '.join(f'I(np.log(carat) * ({g} == {level!r}))' for g in groups for level in category_levels[g][1:])\n    interaction=interaction_terms(list(category_levels))")
    source=source.replace("+' + np.log(carat):('+' + '.join('C('+g+')' for g in remaining)+')'", "+' + '+interaction_terms(remaining)")
    source=source.replace("        diagnostic_models[name]=fitted", "        assert fitted.model.rank == fitted.model.exog.shape[1], 'ランク欠損モデルは解釈しない'\n        diagnostic_models[name]=fitted")
    source=source.replace("        delta=diagnostic_records", "        assert reduced_fit.model.rank == reduced_fit.model.exog.shape[1], 'ランク欠損モデルは解釈しない'\n        delta=diagnostic_records")
    nb.cells[13].source=source
    nb.metadata['execution_notes'].append({'category':'analysis-model','cell':13,'warning':'np.log(carat):C(...)の全レベル項が主スプラインの線形成分と重複しランク欠損','resolution':'各カテゴリの参照レベルを明示的に除いた対数重量×指示変数を生成しランク確認。Jupytermind起因ではない。'})
write_notebook(fix_interactions)
print('交互作用の共線性を参照レベル除外で解消し、ランク検査を追加')

交互作用の共線性を参照レベル除外で解消し、ランク検査を追加


In [20]:
baseline={'sha256':provenance['sha256'],'rows':len(df),'size_log_R2':model_scores[0]['test_log_R2'],'full_log_R2':full_r2,'stratified_Fair_pct':strata_results[0]['ideal_relative_pct'],'stratified_Premium_pct':strata_results[1]['ideal_relative_pct'],'interaction_log_R2':diagnostic_records[1]['log_R2'],'geometry_log_R2':diagnostic_records[2]['log_R2']}
(data_dir/'initial-verification-metrics.json').write_text(json.dumps(baseline,ensure_ascii=False,indent=2))
import base64
before_nb=nbformat.read(handle.notebook_path,as_version=4)
before_images=[o for o in before_nb.cells[7].outputs if 'image/png' in o.get('data',{})]
(data_dir/'layout-before-model-performance.png').write_bytes(base64.b64decode(before_images[3].data['image/png']))
extra_charts_code='''with phase('反復結果・追加図表'):
    extra_chart_specs=[('stratified-cut',strata_table,'bar','comparator','ideal_relative_pct','同一重量・色・透明度層の比較（共通支持のみ）','Idealと比較する等級','幾何平均価格差（%）','ideal_relative_pct'),('heterogeneous-cut',weight_contrast_table,'line','carat','ideal_vs_fair_pct','交互作用モデル：cut差は重量で変化する','重量（ct）','Ideal対Fairのモデル価格差（%）','ideal_vs_fair_pct')]
    extra_rendered_charts=[]
    for chart_id,frame,kind,x,y,title,xlabel,ylabel,legend in extra_chart_specs:
        with warnings.catch_warnings(record=True) as observed_warnings:
            warnings.simplefilter('always')
            png=visualization.render_chart(frame,kind=kind,x=x,y=y,title=title,xlabel=xlabel,ylabel=ylabel)
        metadata={'title':title,'xlabel':xlabel,'ylabel':ylabel,'legend':legend,'missing_glyphs':[str(w.message) for w in observed_warnings if 'Glyph' in str(w.message)],'chart_id':chart_id}
        display(visualization.build_image_output(png)['data'],raw=True)
        extra_rendered_charts.append(metadata)
    print('chart_metadata:',json.dumps(extra_rendered_charts,ensure_ascii=False))
'''
probe_code='''with phase('Jupytermind改善候補・再現確認'):
    from dataclasses import replace
    probe_handle=replace(handle,notebook_path=data_dir/'temporary-api-probe.ipynb')
    lifecycle.mark_write_start(RUN_ID)
    try:
        project_manager.ensure_notebook(probe_handle)
        def seed_probe(nb):
            nb.cells=[nbformat.v4.new_code_cell('print(0.5)',execution_count=1,outputs=[nbformat.v4.new_output('stream',name='stdout',text='0.5\\n')])]
        project_manager.enqueue_write(probe_handle,seed_probe)
    finally:
        lifecycle.mark_write_end(RUN_ID)
    evidence_stream_issue=None
    try:
        insight_engine.record_insight(probe_handle,'probe',1,'0.5','associational',language='ja')
    except insight_engine.EvidenceMissingError as exc:
        evidence_stream_issue={'classification':'機能不足・MCP出力互換性','expected':'正常に実行されたJupyter stream出力も数値根拠として認識','actual':type(exc).__name__,'impact':'printで得た値を根拠登録できない','workaround':'display_data/execute_resultのtext/plainとして値を保存','cell':18,'log':str(exc)}
    visual_probe=nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell('chart_probe',execution_count=1,outputs=[visualization.build_image_output(png)])])
    visual_without_metadata=notebook_audit.audit_visual_outputs(visual_probe,(0,))
    module_probe_results={'stream_evidence':evidence_stream_issue,'visual_missing_metadata_findings':[asdict(f) for f in visual_without_metadata],'layout_before_artifact':str(data_dir/'layout-before-model-performance.png'),'layout_workaround':'matplotlib rcParams figure.autolayout=True'}
    print(json.dumps(module_probe_results,ensure_ascii=False,indent=2))
    (data_dir/'jupytermind-improvement-probes.json').write_text(json.dumps(module_probe_results,ensure_ascii=False,indent=2))
    probe_handle.notebook_path.unlink()
'''
modules_md='''## 使用したJupytermindモジュール

直接import・呼出しのみを列挙。下記の接頭辞はすべて`ai_data_scientist.`。内部依存を使用済みとして数えない。

| モジュール | 直接使用した関数・型 | 使用目的 |
|---|---|---|
| project_manager | resolve_project, ensure_notebook, ensure_data_dir, enqueue_write | 指定保存先の解決、データ/Notebook作成、直列書込み |
| lifecycle | register_run, mark_execution_start/end, mark_write_start/end, is_cancel_requested, mark_completed, mark_failed, get_run_status, wait_for_quiescence | run_idと実行・書込み・終了状態管理 |
| language_router | detect_language | 日本語指定の確認 |
| ingestion | SourceSpec, ingest | 保存CSVの読込、100,000行上限と非切捨て確認 |
| cleaning | clean_dataset | 欠損除去の影響報告、行番号を除いた重複計数/感度分析 |
| eda | explore | 欠損、型、要約、カテゴリ件数 |
| stats_analysis | correlation | carat–priceのPearson相関と日本語解釈 |
| visualization | render_chart, build_image_output | 日本語図8枚とPNG MIME出力 |
| insight_engine | extract_cited_value, record_insight, EvidenceMissingError | 実行済み出力から正確な値を抽出、根拠付きInsight、互換性再現確認 |
| notebook_audit | audit_notebook, audit_visual_outputs | 実行/根拠/visual_audit=True監査とメタデータ欠落の再現 |
| data_definition | FieldValue, build_manifest, manifest.unresolved_fields | 単位・定義・出典の確信度記録と不明/推定の表面化 |
| analysis_assumptions | Assumption, AnalysisAssumptionManifest, check_manifest | 関連の範囲、仮定リスク、検証済み仮定の記録 |
| data_quality | detect_anomalies | 0寸法、不合理範囲、カテゴリ等の意味的検査 |
| sensitivity | SensitivityPlan, run_sensitivity | 2モデル×4前処理の感度分析、相対偏差20%判定 |

### 非適用・使用しなかった機能
mcp_gatewayはimport・呼出ししていない。ホストのJupyter MCPセル実行を直接使用し、カーネルにMCPクライアントが注入されない環境でローカルexecをMCP実行と偽らない。独立した参照データセットがないためdata_quality.validate_anomaliesとdataset_validation.compare_datasetsは非適用。anomaly_detectionの正規z-score検査は右裾が長い価格を誤り扱いしかねないため用いず、宣言制約・多列整合性・除外感度を使用。因果推定は時点/販売過程/介入/識別戦略がないため非適用。SDKによる公開データのカタログ検索はKaggleApiで行い、Jupytermindのデータセット比較機能による自動発見とは区別する。
'''
improvements_md='''## Jupytermind改善候補

| 分類 | 再現条件 | 期待する挙動 | 実際の挙動 | 影響 | 回避策 | 関係セル・ログ |
|---|---|---|---|---|---|---|
| 可視化・レイアウト不足 | render_chartの既定設定で日本語カテゴリの縦向きtickを持つbar図を保存 | 下端のtick/軸ラベルまで収まる | 初回のモデル比較図等で下端が切れた | 図の読み取りが困難 | figure.autolayout=Trueを呼出し前に設定 | セル7、data/layout-before-model-performance.png、セル15の修正版図 |
| 可視化監査・検査不足 | PNGを含む実行済みコードセルのmetadata.chartを空にしてaudit_visual_outputs | 必須のglyph/title/axis/legend監査メタデータ欠落を警告 | findingsが空になる | メタデータが欠けた図を監査合格と誤認し得る | 呼出し側で全画像のラベル/字形warningを検査しセルと出力双方に明示metadataを保存 | セル18、data/jupytermind-improvement-probes.json |
| 根拠エンジン・出力互換性不足 | 実行済みJupyterセルの数値をstdout streamに置きrecord_insight | stream内の数値も根拠として検出 | EvidenceMissingErrorになる（エラー全文は再現ログに保存） | print値のみを根拠とするInsightを記録できない | display_data text/plainにもJSON値を表示し、その実出力から抽出 | セル4/6/9/11/13の数値出力、セル18の最小再現、同JSONログ |

いずれもライブラリのAPI/保存/監査仕様に対応する候補で、Kaggle認証/APIやベンチマークランナーの不具合ではない。MCPが標準streamを生成すること自体は正常。CLIでの長出力省略は分析結果の欠落ではなくNotebookには全出力が残る。最初の辞書assertion（carats/car​atの表記差）と交互作用のランク欠損は分析コード側の問題として修正済みで、Jupytermind改善候補には含めない。修正履歴はNotebook metadata.execution_notes。
'''
final_code='''with phase('結果・根拠リンク・監査準備'):
    baseline=json.loads((data_dir/'initial-verification-metrics.json').read_text())
    actual={'sha256':provenance['sha256'],'rows':len(df),'size_log_R2':model_scores[0]['test_log_R2'],'full_log_R2':full_r2,'stratified_Fair_pct':strata_results[0]['ideal_relative_pct'],'stratified_Premium_pct':strata_results[1]['ideal_relative_pct'],'interaction_log_R2':diagnostic_records[1]['log_R2'],'geometry_log_R2':diagnostic_records[2]['log_R2']}
    for key,value in actual.items():
        if isinstance(value,float):
            assert np.isclose(value,baseline[key],rtol=1e-9,atol=1e-9),(key,value,baseline[key])
        else:
            assert value==baseline[key],(key,value,baseline[key])
    current_nb=nbformat.read(handle.notebook_path,as_version=4)
    counts=[c.execution_count for c in current_nb.cells if c.cell_type=='code' and not c.source.startswith("with phase('結果・根拠")]
    assert all(isinstance(c,int) for c in counts) and counts==sorted(counts) and len(counts)==len(set(counts)), '全セルの上からの実行順が不整合'
    def attach_chart_metadata(nb):
        for index,metadata_list in [(7,rendered_charts),(15,extra_rendered_charts)]:
            image_outputs=[o for o in nb.cells[index].outputs if 'image/png' in o.get('data',{})]
            assert len(image_outputs)==len(metadata_list)
            for output,metadata in zip(image_outputs,metadata_list):
                assert not metadata['missing_glyphs'],metadata
                assert all(metadata.get(k) for k in ['title','xlabel','ylabel','legend'])
                output.setdefault('metadata',{})['chart']=metadata
            nb.cells[index].metadata['charts']=metadata_list
            nb.cells[index].metadata['chart']={key:' / '.join(m[key] for m in metadata_list) for key in ['title','xlabel','ylabel','legend']}
            nb.cells[index].metadata['chart']['missing_glyphs']=[]
        nb.metadata['reexecution']={'kernel_restarted_via':'jupyter-restart_notebook','all_preceding_code_counts':counts,'metrics_equal_to_first_analysis':True,'comparison_rtol':1e-9,'comparison_atol':1e-9,'data_sha256':actual['sha256']}
    write_notebook(attach_chart_metadata)
    evidence_sources=[(4,'pearson'),(6,'size_only_log_R2'),(6,'size_only_log_R2'),(9,'stratified_results'),(11,'sensitivity_summary'),(13,'diagnostic_summary')]
    texts=[
        f'Insight 1｜大きさが価格差の主軸。caratとpriceのPearson相関は{correlation_result.statistic:.4f}、Spearmanは{spearman:.4f}。ただし相関は因果を示さない。寸法ゼロ/20mm超が23件あり、主モデルは寸法に依存させず、行番号を除いた146件の重複も感度分析で扱った。',
        f'Insight 2｜固定holdoutで重量のみの対数価格R2は{model_scores[0]["test_log_R2"]:.4f}、品質追加は{model_scores[1]["test_log_R2"]:.4f}、比率追加は{full_r2:.4f}。品質を加えるとUSD MAEは約{model_scores[0]["test_USD_MAE_smearing"]:.0f}から{model_scores[1]["test_USD_MAE_smearing"]:.0f}へ低下した。群除外と反復3でも透明度、色、カットの順に独自の追加説明力が大きい。R2は観察的・対数尺度の説明力であり、因果的寄与率ではない。',
        f'Insight 3｜カットの単純な価格順位には大きさの交絡がある。Idealの未調整中央値は{cut_raw.loc["Ideal","median_price_USD"]:.0f}、Fairは{cut_raw.loc["Fair","median_price_USD"]:.0f}で、平均重量はそれぞれ{cut_raw.loc["Ideal","mean_carat"]:.3f}ctと{cut_raw.loc["Fair","mean_carat"]:.3f}ct。初回加法モデルで1ct・G・VS2・共通depth/tableに固定するとIdealはFairより約{adjusted_table.query("feature == \'cut\' and level == \'Ideal\'").relative_pct.iloc[0]:.1f}%高い。ただし反復3からこの差を全重量に一律適用してはならない。',
        f'Insight 4／反復1結果｜carat/color/clarity完全一致層ではIdeal対Fairの価格差は{strata_results[0]["ideal_relative_pct"]:.2f}%（108層）、対Premiumは{strata_results[1]["ideal_relative_pct"]:.2f}%（1,073層）。Fair比較は対象2等級の{strata_results[0]["subset_coverage_pct"]:.2f}%だけの共通支持なので全体の差とは異なる。層bootstrap区間と件数はセル9に保存。depth/tableや売主等の交絡、丸められた重量、観測選択が残り、因果効果とは呼ばない。',
        f'Insight 5／反復2結果｜8仕様・共通holdout {len(common_eval):,}件で品質追加のR2増分は全て正。quality_gain stable={quality_sensitivity.stable}、max_relative_deviation={quality_sensitivity.max_relative_deviation:.6f}。Ideal対Fair差は{cut_sensitivity.stable}、max_relative_deviation={cut_sensitivity.max_relative_deviation:.6f}（判定閾値0.20）、約{sensitivity_table.ideal_vs_fair_pct.min():.2f}–{sensitivity_table.ideal_vs_fair_pct.max():.2f}%。除外・関数形に対する方向と大きさはこの範囲では維持。ただし共通評価は中心域であり、加法モデルの比較であって重量別差の一定性を保証しない。',
        f'Insight 6／反復3結果｜品質×重量の交互作用による対数R2増分は{diagnostic_summary["interaction_gain_log_R2"]:.6f}、寸法の追加は{diagnostic_summary["dimension_gain_log_R2"]:.6f}。重量・品質が既知なら寸法の独自情報は小さい。交互作用モデルのIdeal対Fair差は0.3ctで{weight_contrasts[0]["ideal_vs_fair_pct"]:.1f}%、1ctで{weight_contrasts[2]["ideal_vs_fair_pct"]:.1f}%、2ctで{weight_contrasts[4]["ideal_vs_fair_pct"]:.1f}%と変化。透明度/色/cutの順位は維持するが、2ct以上の残差偏りと少数例が残る。加法の約17%は参照条件の要約に限定し、市場全体の因果的premiumとはしない。最大3回の反復で終了。'
    ]
    for slot,(cell_index,key),text in zip(range(20,26),evidence_sources,texts):
        evidence_nb=nbformat.read(handle.notebook_path,as_version=4)
        source_cell=evidence_nb.cells[cell_index]
        output_text='\\n'.join(str(o.get('data',{}).get('text/plain','')) for o in source_cell.outputs)
        cited=insight_engine.extract_cited_value({'output':output_text},r'(\\{"'+key+r'":.*\\})')
        lifecycle.mark_write_start(RUN_ID)
        try:
            insight_engine.record_insight(handle,text,source_cell.execution_count,cited,'associational',language='ja')
        finally:
            lifecycle.mark_write_end(RUN_ID)
        def place_insight(nb,target=slot):
            recorded=nb.cells.pop()
            assert recorded.cell_type=='markdown' and '```evidence' in recorded.source
            recorded.metadata['generated_insight']=True
            nb.cells[target]=recorded
        write_notebook(place_insight)
    final_assumptions=analysis_assumptions.AnalysisAssumptionManifest(analysis_scope=assumed_manifest.analysis_scope,assumptions=assumed_manifest.assumptions+(analysis_assumptions.Assumption('preprocessing-and-form', '重複/異常/トリム・線形/スプラインの8仕様で主結論を検証', 'tested',evidence_cell=11,conclusion_critical=True),analysis_assumptions.Assumption('constant-cut-premium', 'カット価格差が全重量で一定という解釈は採用しない', 'rejected',evidence_cell=13,conclusion_critical=False)),causal_scope='associational',sampling=assumed_manifest.sampling)
    final_findings=analysis_assumptions.check_manifest(final_assumptions)
    print('残る前提警告:',json.dumps([asdict(f) for f in final_findings],ensure_ascii=False))
    (data_dir/'analysis-assumptions-manifest.json').write_text(json.dumps(asdict(final_assumptions),ensure_ascii=False,indent=2))
    (data_dir/'verified-reexecution-metrics.json').write_text(json.dumps(actual,ensure_ascii=False,indent=2))

def persist_post_execution_audit():
    report=notebook_audit.audit_notebook(handle.notebook_path,visual_audit=True)
    if not report.ok:
        lifecycle.mark_failed(RUN_ID)
        print(json.dumps(asdict(report),ensure_ascii=False,indent=2))
        raise RuntimeError('Notebook監査未合格')
    lifecycle.mark_completed(RUN_ID)
    def save_audit(nb):
        nb.metadata['notebook_audit']={**asdict(report),'ok':report.ok,'visual_audit':True,'checked_at_utc':datetime.now(timezone.utc).isoformat()}
        nb.metadata['lifecycle']=asdict(lifecycle.get_run_status(RUN_ID))
        nb.metadata['phase_log']=phase_log
        nb.metadata['final_report']={'rows':len(df),'provenance':provenance,'model_scores':model_scores,'sensitivity':sensitivity_summary,'diagnostics':diagnostic_summary,'remaining_assumption_findings':[asdict(f) for f in final_findings]}
    write_notebook(save_audit)
    status=lifecycle.wait_for_quiescence(RUN_ID)
    # 書込み中ではなく、終了後のスナップショットをNotebookへ記録。
    project_manager.enqueue_write(handle,lambda nb:nb.metadata.update({'lifecycle':asdict(status),'phase_log':phase_log}))
    (data_dir/'lifecycle-and-audit.json').write_text(json.dumps({'audit':{**asdict(report),'ok':report.ok,'visual_audit':True},'lifecycle':asdict(status),'phase_log':phase_log},ensure_ascii=False,indent=2))
    print(json.dumps({'audit_ok':report.ok,'executed_code_cells':report.executed_code_cell_count,'code_cells':report.code_cell_count,'visual_findings':[asdict(f) for f in report.visual_findings],'insights':report.insight_cell_count,'lifecycle':asdict(status),'reexecution_metrics_equal':True},ensure_ascii=False))
    return report.ok
persist_post_execution_audit()
'''
# sensitivityの「品質追加」は形状比率追加と混ぜずに計測する。
def append_final(nb):
    nb.cells[11].source=nb.cells[11].source.replace("            extended=smf.ols", "            quality_only=smf.ols('np.log(price) ~ '+size+'+C(cut)+C(color)+C(clarity)',data=selected_train).fit()\n            quality_r2=r2_score(np.log(common_eval.price),quality_only.predict(common_eval))\n            extended=smf.ols")
    nb.cells[11].source=nb.cells[11].source.replace("'quality_gain_R2':extended_r2-base_r2", "'quality_gain_R2':quality_r2-base_r2")
    nb.cells.extend([nbformat.v4.new_markdown_cell('## 反復結果の図表\n\n共通支持の層比較と、交互作用による重量別差を別々に表示する。尺度・対象が違うため同じ母集団の推定値として混同しない。'),nbformat.v4.new_code_cell(extra_charts_code),nbformat.v4.new_markdown_cell(modules_md),nbformat.v4.new_markdown_cell(improvements_md),nbformat.v4.new_code_cell(probe_code),nbformat.v4.new_markdown_cell('## 根拠付きInsightと残る限界\n\n下記は最終セルが実行済み数値から自動更新する。USD/ct/mmは公開ggplot2辞書との対応に基づく推定。Kaggle独自の測定/価格時点/市場代表性は不明で、単位対応とレコードの独立性に関するassumption警告を残す。記載のR2は固定holdout内の観察的説明力で、探索にholdoutを再利用しているため外部データに対する保証にはならない。寸法20mmやdepth差2pp等は検査閾値で、真の誤入力とは断定しない。')])
    nb.cells.extend([nbformat.v4.new_markdown_cell('### Insight更新待ち '+str(i+1)) for i in range(6)])
    nb.cells.append(nbformat.v4.new_code_cell(final_code))
    nb.metadata['first_analysis_metrics']=baseline
write_notebook(append_final)
print({'cells':len(nbformat.read(handle.notebook_path,as_version=4).cells),'baseline':baseline,'final_code_index':26})

{'cells': 27, 'baseline': {'sha256': 'fc2f171cc18eae2138d01dcca7179db3bb30ff047dceae4467a056d52133810a', 'rows': 53940, 'size_log_R2': 0.9356245415575064, 'full_log_R2': 0.9849644665031242, 'stratified_Fair_pct': 22.209576722213, 'stratified_Premium_pct': 1.1019600858798366, 'interaction_log_R2': 0.9855981452378861, 'geometry_log_R2': 0.985740567544255}, 'final_code_index': 26}


In [21]:
import ast
for i,cell in enumerate(nbformat.read(handle.notebook_path,as_version=4).cells):
    if cell.cell_type=='code':
        try:
            ast.parse(cell.source)
        except SyntaxError as exc:
            print({'cell':i,'syntax_error':str(exc)})
print('コード構文の事前確認終了')

コード構文の事前確認終了
